# TP4 — Knowledge Base Construction, Alignment & Expansion
**Domain : Science-Fiction (authors & works)**

In [4]:
# ── Imports (regroupés ici une seule fois) ──
import requests
import urllib.parse
import time
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD

# ── Namespaces ──
PRIV = Namespace("http://myproject.org/resource/")   # entités
PRED = Namespace("http://myproject.org/predicate/")  # prédicats privés
DBO  = Namespace("http://dbpedia.org/ontology/")     # DBpedia

# ── Graphe principal (utilisé dans toutes les cellules suivantes) ──
my_private_kg = Graph()
my_private_kg.bind("priv", PRIV)
my_private_kg.bind("pred", PRED)
my_private_kg.bind("dbo",  DBO)

print("Graphe initialisé.")
print(f"Triplets : {len(my_private_kg)}")

Graphe initialisé.
Triplets : 0


In [5]:
# ── Step 1 : KB initiale via Open Library API ──

def fetch_books_from_api(subject="science_fiction", limit=35):
    url = f"https://openlibrary.org/subjects/{subject}.json?limit={limit}"
    print(f"Fetching: {url}")
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        return response.json().get("works", [])
    except Exception as e:
        print(f"Erreur API : {e}")
        return []

def build_initial_kg(books_data, kg):
    # On enrichit le graphe existant (pas un nouveau)
    for book in books_data:
        # Entité livre
        raw_title = book.get("title", "Unknown")
        safe_title = urllib.parse.quote(raw_title.replace(" ", "_"))
        book_uri = PRIV[safe_title]

        kg.add((book_uri, RDF.type,    PRIV.Book))
        kg.add((book_uri, PRED.title,  Literal(raw_title, datatype=XSD.string)))

        if "first_publish_year" in book:
            kg.add((book_uri, PRED.firstPublishYear,
                    Literal(book["first_publish_year"], datatype=XSD.integer)))

        # Entités auteurs
        for author_data in book.get("authors", []):
            raw_name  = author_data.get("name", "Unknown")
            safe_name = urllib.parse.quote(raw_name.replace(" ", "_"))
            author_uri = PRIV[safe_name]

            kg.add((author_uri, RDF.type,       PRIV.Author))
            kg.add((author_uri, PRED.wroteBook, book_uri))
            kg.add((author_uri, PRED.name,      Literal(raw_name, datatype=XSD.string)))

    return kg

# Exécution
books = fetch_books_from_api()
my_private_kg = build_initial_kg(books, my_private_kg)

# Stats
all_uris = set(my_private_kg.subjects()) | {o for o in my_private_kg.objects() if isinstance(o, URIRef)}
print(f"Triplets  : {len(my_private_kg)}")
print(f"Entités   : {len(all_uris)}")

Fetching: https://openlibrary.org/subjects/science_fiction.json?limit=35
Triplets  : 188
Entités   : 61


In [12]:
# ── Step 2 : Entity Linking via DBpedia SPARQL ──

def link_entity_sparql(wiki_url):
    # Dérive l'URI DBpedia depuis l'URL Wikipedia
    page_name   = wiki_url.split("/wiki/")[-1]
    dbpedia_uri = f"http://dbpedia.org/resource/{page_name}"

    # Vérifie que l'URI existe bien dans DBpedia
    url   = "https://dbpedia.org/sparql"
    query = f"ASK {{ <{dbpedia_uri}> ?p ?o . }}"
    params = {"query": query, "format": "application/json"}

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        exists = r.json().get("boolean", False)
        if exists:
            return dbpedia_uri, 1.0
    except Exception as e:
        print(f"Erreur pour {dbpedia_uri} : {e}")
    return None, 0.0

# Les mêmes URLs que TD1
wiki_urls = [
    "https://en.wikipedia.org/wiki/Isaac_Asimov",
    "https://en.wikipedia.org/wiki/Frank_Herbert",
    "https://en.wikipedia.org/wiki/Arthur_C._Clarke",
    "https://en.wikipedia.org/wiki/Philip_K._Dick",
    "https://en.wikipedia.org/wiki/Ursula_K._Le_Guin",
    "https://en.wikipedia.org/wiki/Science_fiction",
    "https://en.wikipedia.org/wiki/Dune_(novel)",
]

mapping_table = []

for wiki_url in wiki_urls:
    page_name   = wiki_url.split("/wiki/")[-1]
    private_uri = PRIV[page_name]

    dbpedia_uri, conf = link_entity_sparql(wiki_url)

    if dbpedia_uri:
        # ✅ Ajout du owl:sameAs dans le graphe
        my_private_kg.add((private_uri, OWL.sameAs, URIRef(dbpedia_uri)))

    mapping_table.append({
        "Private Entity": page_name,
        "External URI":   dbpedia_uri or "Not Found",
        "Confidence":     conf
    })

# Tableau de mapping (livrable du TD)
print(f"{'Private Entity':<25} | {'External URI':<55} | Confidence")
print("-" * 95)
for row in mapping_table:
    print(f"{row['Private Entity']:<25} | {row['External URI']:<55} | {row['Confidence']}")

print(f"\nowl:sameAs ajoutés : {len(list(my_private_kg.triples((None, OWL.sameAs, None))))}")

Private Entity            | External URI                                            | Confidence
-----------------------------------------------------------------------------------------------
Isaac_Asimov              | http://dbpedia.org/resource/Isaac_Asimov                | 1.0
Frank_Herbert             | http://dbpedia.org/resource/Frank_Herbert               | 1.0
Arthur_C._Clarke          | http://dbpedia.org/resource/Arthur_C._Clarke            | 1.0
Philip_K._Dick            | http://dbpedia.org/resource/Philip_K._Dick              | 1.0
Ursula_K._Le_Guin         | http://dbpedia.org/resource/Ursula_K._Le_Guin           | 1.0
Science_fiction           | http://dbpedia.org/resource/Science_fiction             | 1.0
Dune_(novel)              | http://dbpedia.org/resource/Dune_(novel)                | 1.0

owl:sameAs ajoutés : 7


In [13]:
# ── Step 2 (suite) : Entité non trouvée → définition ontologique ──
# Exemple : un fanzine Sci-Fi, qui n'existe pas dans DBpedia

# 1. Nouvelle classe
my_private_kg.add((PRIV.SciFiFanzine, RDFS.subClassOf, DBO.PeriodicalLiterature))

# 2. Instance de cette classe
my_private_kg.add((PRIV.GalaxyMagazine1950, RDF.type,       PRIV.SciFiFanzine))
my_private_kg.add((PRIV.GalaxyMagazine1950, PRED.title,     Literal("Galaxy Science Fiction", datatype=XSD.string)))
my_private_kg.add((PRIV.GalaxyMagazine1950, PRED.foundedIn, Literal(1950, datatype=XSD.integer)))

# 3. Définition des propriétés (domain / range)
my_private_kg.add((PRED.foundedIn, RDFS.domain, PRIV.SciFiFanzine))
my_private_kg.add((PRED.foundedIn, RDFS.range,  XSD.integer))

# Vérification
print("Triplets ajoutés pour l'entité personnalisée :")
for s, p, o in my_private_kg.triples((PRIV.GalaxyMagazine1950, None, None)):
    print(f"  {s.split('/')[-1]} → {p.split('/')[-1]} → {o}")

print(f"\nTotal triplets dans my_private_kg : {len(my_private_kg)}")

Triplets ajoutés pour l'entité personnalisée :
  GalaxyMagazine1950 → 22-rdf-syntax-ns#type → http://myproject.org/resource/SciFiFanzine
  GalaxyMagazine1950 → title → Galaxy Science Fiction
  GalaxyMagazine1950 → foundedIn → 1950

Total triplets dans my_private_kg : 201


In [14]:
# ── Step 3 : Alignement de prédicats via SPARQL ──

def search_dbo_properties(keyword):
    url = "https://dbpedia.org/sparql"
    query = f"""
    PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?property ?label WHERE {{
        ?property a rdf:Property .
        FILTER(STRSTARTS(STR(?property), "http://dbpedia.org/ontology/"))
        FILTER(CONTAINS(LCASE(STR(?property)), LCASE("{keyword}")))
        OPTIONAL {{ ?property rdfs:label ?label . FILTER(LANG(?label) = "en") }}
    }}
    LIMIT 10
    """
    params = {"query": query, "format": "application/json"}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        results = []
        for b in r.json()["results"]["bindings"]:
            results.append({
                "URI":   b["property"]["value"],
                "Label": b.get("label", {}).get("value", "(no label)")
            })
        return results
    except Exception as e:
        print(f"Erreur : {e}")
        return []

# On cherche les candidats pour nos 3 prédicats privés
for keyword in ["author", "year", "title"]:
    results = search_dbo_properties(keyword)
    print(f"\nKeyword '{keyword}' → {len(results)} candidat(s) :")
    for c in results[:3]:
        print(f"  {c['URI']}")
        print(f"  Label : {c['Label']}")


Keyword 'author' → 7 candidat(s) :
  http://dbpedia.org/ontology/author
  Label : author
  http://dbpedia.org/ontology/authority
  Label : authority
  http://dbpedia.org/ontology/unitaryAuthority
  Label : unitary authority

Keyword 'year' → 10 candidat(s) :
  http://dbpedia.org/ontology/statisticYear
  Label : statistic year
  http://dbpedia.org/ontology/activeYears
  Label : active years
  http://dbpedia.org/ontology/activeYearsEndDate
  Label : active years end date

Keyword 'title' → 10 candidat(s) :
  http://dbpedia.org/ontology/chairmanTitle
  Label : chairman title
  http://dbpedia.org/ontology/leaderTitle
  Label : leader title
  http://dbpedia.org/ontology/managerTitle
  Label : manager title


In [15]:
for keyword in ["name", "work", "genre"]:
    results = search_dbo_properties(keyword)
    print(f"\nKeyword '{keyword}' → {len(results)} candidat(s) :")
    for c in results[:3]:
        print(f"  {c['URI']}")
        print(f"  Label : {c['Label']}")


Keyword 'name' → 10 candidat(s) :
  http://dbpedia.org/ontology/fuelTypeName
  Label : fuel type
  http://dbpedia.org/ontology/alternativeName
  Label : alternative name
  http://dbpedia.org/ontology/birthName
  Label : birth name

Keyword 'work' → 10 candidat(s) :
  http://dbpedia.org/ontology/formerBroadcastNetwork
  Label : former broadcast network
  http://dbpedia.org/ontology/broadcastNetwork
  Label : broadcast network
  http://dbpedia.org/ontology/network
  Label : network

Keyword 'genre' → 5 candidat(s) :
  http://dbpedia.org/ontology/genre
  Label : genre
  http://dbpedia.org/ontology/literaryGenre
  Label : literary genre
  http://dbpedia.org/ontology/musicFusionGenre
  Label : music fusion genre


In [16]:
# ── Step 3 (suite) : Application des alignements de prédicats ──

# 1. Exact match
my_private_kg.add((PRED.title, OWL.equivalentProperty, DBO.title))
my_private_kg.add((PRED.genre, OWL.equivalentProperty, DBO.genre))

# 2. Sous-propriétés (plus spécifique que DBpedia)
my_private_kg.add((PRED.firstPublishYear, RDFS.subPropertyOf, DBO.activeYears))
my_private_kg.add((PRED.name,            RDFS.subPropertyOf, DBO.alternativeName))

# 3. Inverse (direction sujet/objet inversée)
my_private_kg.add((PRED.wroteBook, OWL.inverseOf, DBO.author))

# ── Vérification ──
print("Equivalent properties :")
for s, _, o in my_private_kg.triples((None, OWL.equivalentProperty, None)):
    print(f"  {s.split('/')[-1]}  ≡  {o.split('/')[-1]}")

print("\nSub-properties :")
for s, _, o in my_private_kg.triples((None, RDFS.subPropertyOf, None)):
    print(f"  {s.split('/')[-1]}  ⊂  {o.split('/')[-1]}")

print("\nInverse properties :")
for s, _, o in my_private_kg.triples((None, OWL.inverseOf, None)):
    print(f"  {s.split('/')[-1]}  ↔  {o.split('/')[-1]}")

print(f"\nTotal triplets dans my_private_kg : {len(my_private_kg)}")

Equivalent properties :
  title  ≡  title
  genre  ≡  genre

Sub-properties :
  firstPublishYear  ⊂  activeYears
  name  ⊂  alternativeName

Inverse properties :
  wroteBook  ↔  author

Total triplets dans my_private_kg : 206


In [17]:
# ── Step 4 : Expansion 2-hop depuis 300 auteurs Sci-Fi ──

def get_scifi_anchors(limit=300):
    url = "https://dbpedia.org/sparql"
    query = f"""
    PREFIX dbo: <http://dbpedia.org/ontology/>
    PREFIX dbr: <http://dbpedia.org/resource/>
    SELECT DISTINCT ?author WHERE {{
        ?author a dbo:Writer .
        ?author dbo:genre dbr:Science_fiction .
    }}
    LIMIT {limit}
    """
    params = {"query": query, "format": "application/json"}
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        return [b["author"]["value"] for b in r.json()["results"]["bindings"]]
    except Exception as e:
        print(f"Erreur : {e}")
        return []

def fetch_2hop(dbpedia_uri):
    url = "https://dbpedia.org/sparql"
    # Prédicats bruités à exclure
    excluded = (
        "dbo:abstract, "
        "<http://dbpedia.org/property/wikiPageUsesTemplate>, "
        "<http://www.w3.org/ns/prov#wasDerivedFrom>, "
        "<http://xmlns.com/foaf/0.1/isPrimaryTopicOf>"
    )
    query = f"""
    PREFIX dbo: <http://dbpedia.org/ontology/>
    CONSTRUCT {{
        <{dbpedia_uri}> ?p ?o .
        ?work ?work_p ?work_o .
    }}
    WHERE {{
        {{ <{dbpedia_uri}> ?p ?o .
           FILTER(?p NOT IN ({excluded})) }}
        UNION
        {{ <{dbpedia_uri}> dbo:notableWork ?work .
           ?work ?work_p ?work_o .
           FILTER(?work_p NOT IN ({excluded})) }}
    }}
    LIMIT 6000
    """
    params = {"query": query, "format": "text/turtle"}
    tmp = Graph()
    try:
        r = requests.get(url, params=params, timeout=20)
        r.raise_for_status()
        tmp.parse(data=r.text, format="turtle")
    except Exception:
        pass  # on continue si une ancre échoue
    return tmp

# Récupération des ancres
anchors = get_scifi_anchors(limit=300)
print(f"{len(anchors)} ancres trouvées. Expansion en cours...\n")

# Expansion
final_kg = Graph()
for idx, anchor in enumerate(anchors, 1):
    if idx % 10 == 0:
        print(f"  {idx}/{len(anchors)} — triplets : {len(final_kg)}")
    final_kg += fetch_2hop(anchor)
    time.sleep(0.1)  # respect du rate limit DBpedia

# Stats finales
all_subjects = set(final_kg.subjects())
all_uri_objects = {o for o in final_kg.objects() if isinstance(o, URIRef)}
all_entities = all_subjects | all_uri_objects

print(f"\n{'='*50}")
print(f"Triplets   : {len(final_kg):>8,}  (cible : 50k–200k)")
print(f"Entités    : {len(all_entities):>8,}  (cible : 5k–30k)")
print(f"Prédicats  : {len(set(final_kg.predicates())):>8,}  (cible : 50–200)")
print(f"{'='*50}")

# Export
final_kg.serialize(destination="final_expanded_kg.nt", format="nt", encoding="utf-8")
print("\nExporté → final_expanded_kg.nt")

300 ancres trouvées. Expansion en cours...

  10/300 — triplets : 1352
  20/300 — triplets : 6494
  30/300 — triplets : 8744
  40/300 — triplets : 10143
  50/300 — triplets : 12559
  60/300 — triplets : 14127
  70/300 — triplets : 15156
  80/300 — triplets : 16340
  90/300 — triplets : 17374
  100/300 — triplets : 19068
  110/300 — triplets : 20096
  120/300 — triplets : 22667
  130/300 — triplets : 25205
  140/300 — triplets : 26556
  150/300 — triplets : 27747
  160/300 — triplets : 30668
  170/300 — triplets : 35017
  180/300 — triplets : 39554
  190/300 — triplets : 41531
  200/300 — triplets : 43563
  210/300 — triplets : 44696
  220/300 — triplets : 46367
  230/300 — triplets : 47215
  240/300 — triplets : 51817
  250/300 — triplets : 58652
  260/300 — triplets : 64111
  270/300 — triplets : 67755
  280/300 — triplets : 71059
  290/300 — triplets : 73299
  300/300 — triplets : 75429

Triplets   :   75,603  (cible : 50k–200k)
Entités    :   31,504  (cible : 5k–30k)
Prédicats  :   

In [18]:
# ── Nettoyage avant export ──
from collections import Counter

# 1. Compter la fréquence de chaque prédicat
pred_counts = Counter(str(p) for _, p, _ in final_kg)
print(f"Prédicats avant nettoyage : {len(pred_counts)}")

# 2. Garder uniquement les 150 prédicats les plus fréquents
top_predicates = {p for p, _ in pred_counts.most_common(150)}

# 3. Reconstruire un graphe propre
clean_kg = Graph()
for s, p, o in final_kg:
    if isinstance(s, URIRef) and isinstance(o, URIRef):  # URI→URI uniquement
        if str(p) in top_predicates:
            clean_kg.add((s, p, o))

# 4. Stats finales
all_entities_clean = set(clean_kg.subjects()) | {o for o in clean_kg.objects() if isinstance(o, URIRef)}

print(f"\n{'='*50}")
print(f"Triplets   : {len(clean_kg):>8,}  (cible : 50k–200k)")
print(f"Entités    : {len(all_entities_clean):>8,}  (cible : 5k–30k)")
print(f"Prédicats  : {len(set(clean_kg.predicates())):>8,}  (cible : 50–200)")
print(f"{'='*50}")

# 5. Export final
clean_kg.serialize(destination="final_expanded_kg.nt", format="nt", encoding="utf-8")
print("\nExporté → final_expanded_kg.nt")

Prédicats avant nettoyage : 394

Triplets   :   57,828  (cible : 50k–200k)
Entités    :   31,481  (cible : 5k–30k)
Prédicats  :       80  (cible : 50–200)

Exporté → final_expanded_kg.nt


In [19]:
# ── Cell 8 : Entity linking depuis extracted_knowledge_scifi.csv ──
import pandas as pd

# 1. Charger les entités extraites en TD1
df = pd.read_csv("extracted_knowledge_scifi.csv")

# Garder uniquement PERSON et WORK_OF_ART
relevant_entities = df[df['type'].isin(['PERSON', 'WORK_OF_ART'])]['entity'].dropna().unique()
print(f"Entités à traiter : {len(relevant_entities)}")

# 2. Linking via DBpedia ASK (même approche que Cell 2)
def link_entity_to_dbpedia(entity_label):
    page_name   = entity_label.replace(" ", "_")
    dbpedia_uri = f"http://dbpedia.org/resource/{page_name}"
    url   = "https://dbpedia.org/sparql"
    query = f"ASK {{ <{dbpedia_uri}> ?p ?o . }}"
    params = {"query": query, "format": "application/json"}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        if r.json().get("boolean", False):
            return dbpedia_uri, 1.0
    except Exception:
        pass
    return None, 0.0

# 3. Boucle de linking + intégration dans clean_kg
mapping_results = []
print("Linking en cours...")

for idx, entity in enumerate(relevant_entities):
    if idx > 0 and idx % 20 == 0:
        print(f"  {idx}/{len(relevant_entities)}")

    uri, conf = link_entity_to_dbpedia(entity)
    mapping_results.append({
        "Private Entity": entity,
        "External URI":   uri or "Not Found",
        "Confidence":     conf
    })

    if uri:
        # ✅ Ajout du owl:sameAs dans clean_kg
        private_uri = PRIV[entity.replace(" ", "_")]
        clean_kg.add((private_uri, OWL.sameAs, URIRef(uri)))

    time.sleep(0.1)  # rate limiting

# 4. Sauvegarde du tableau de mapping
df_mapping = pd.DataFrame(mapping_results)
df_mapping.to_csv("entity_alignment.csv", index=False, encoding="utf-8")

# 5. Stats
found = df_mapping[df_mapping["Confidence"] > 0]
print(f"\nEntités alignées : {len(found)}/{len(relevant_entities)}")
print(df_mapping.head(10).to_string(index=False))

# 6. Re-export du graphe final enrichi
clean_kg.serialize(destination="final_expanded_kg.nt", format="nt", encoding="utf-8")
print(f"\nGraphe final : {len(clean_kg)} triplets")
print("Exporté → final_expanded_kg.nt")
print("Mapping  → entity_alignment.csv")

Entités à traiter : 236
Linking en cours...
  20/236
  40/236
  60/236
  80/236
  100/236
  120/236
  140/236
  160/236
  180/236
  200/236
  220/236

Entités alignées : 168/236
         Private Entity                                   External URI  Confidence
           Isaac Asimov       http://dbpedia.org/resource/Isaac_Asimov         1.0
             Petrovichi         http://dbpedia.org/resource/Petrovichi         1.0
                    PhD                http://dbpedia.org/resource/PhD         1.0
Charles Reginald Dawson                                      Not Found         0.0
      Robert Elderfield  http://dbpedia.org/resource/Robert_Elderfield         1.0
                 Asimov             http://dbpedia.org/resource/Asimov         1.0
     Robert A. Heinlein http://dbpedia.org/resource/Robert_A._Heinlein         1.0
    Arthur C. Clarke.[2                                      Not Found         0.0
                  Bible              http://dbpedia.org/resource/Bible     